In [98]:
import pandas as pd

In [99]:
df = pd.read_csv("data_with_elo.csv", index_col=0)

In [100]:
df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

In [101]:
df["win"] = df["win"].astype(bool)

In [102]:
df["game_date"] = df["gameDateTimeEst"].dt.date
df["last_game_played"] = df.groupby("teamName")["game_date"].shift(1)

In [103]:
df['game_date'] = pd.to_datetime(df['game_date'])
df['last_game_played'] = pd.to_datetime(df['last_game_played'])

In [104]:
df["days_rest"] = (df["game_date"] - df["last_game_played"]).dt.days
df["is_B2B"] = df["days_rest"] == 1

In [105]:
df = df.sort_values(by=["gameDateTimeEst", "gameId"])

In [106]:
rolling_features = ['possessions', 'eFG', 'TO%',
       'OREB%', 'FTR', 'off_rating', 'def_rating', 'net_rating']


for feature in rolling_features:
       df[f"{feature}_rolling"] = df.groupby('teamName')[f"{feature}"].transform(lambda x: x.ewm(span=10).mean())

In [107]:
ewma_cols = [f"{f}_rolling" for f in rolling_features]
df[ewma_cols] = df.groupby('teamName')[ewma_cols].shift(1)

In [108]:
df = df.sort_values(by=["gameDateTimeEst"])

In [109]:
df = df.drop(columns=["points","opponentScore", "teamId", "season"])
df = df.drop(columns=rolling_features)

In [110]:
home_df = df[df["home"] == 1]
away_df = df[df["home"] == 0]

game_df = pd.merge(home_df, away_df, on='gameId', suffixes=('_home', '_away'))

In [111]:
game_df.columns

Index(['gameId', 'gameDateTimeEst_home', 'teamName_home', 'home_home',
       'win_home', 'pre_game_elo_home', 'game_date_home',
       'last_game_played_home', 'days_rest_home', 'is_B2B_home',
       'possessions_rolling_home', 'eFG_rolling_home', 'TO%_rolling_home',
       'OREB%_rolling_home', 'FTR_rolling_home', 'off_rating_rolling_home',
       'def_rating_rolling_home', 'net_rating_rolling_home',
       'gameDateTimeEst_away', 'teamName_away', 'home_away', 'win_away',
       'pre_game_elo_away', 'game_date_away', 'last_game_played_away',
       'days_rest_away', 'is_B2B_away', 'possessions_rolling_away',
       'eFG_rolling_away', 'TO%_rolling_away', 'OREB%_rolling_away',
       'FTR_rolling_away', 'off_rating_rolling_away',
       'def_rating_rolling_away', 'net_rating_rolling_away'],
      dtype='str')

In [112]:
game_df

,gameId,gameDateTimeEst_home,teamName_home,home_home,win_home,pre_game_elo_home,game_date_home,last_game_played_home,days_rest_home,is_B2B_home,...,days_rest_away,is_B2B_away,possessions_rolling_away,eFG_rolling_away,TO%_rolling_away,OREB%_rolling_away,FTR_rolling_away,off_rating_rolling_away,def_rating_rolling_away,net_rating_rolling_away
0,28600009,1986-10-31 20:00:00,Suns,1,True,1444.3523,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,28600008,1986-10-31 20:00:00,Mavericks,1,True,1547.2805,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,28600007,1986-10-31 20:00:00,Nuggets,1,True,1514.6084,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28600006,1986-10-31 20:00:00,Pistons,1,False,1532.8929,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,28600005,1986-10-31 20:00:00,Kings,1,True,1461.1960,1986-10-31,NaT,NaN,False,...,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51330,42500207,2026-05-17 20:00:00,Pistons,1,False,1691.9800,2026-05-17,2026-05-15,2.0,False,...,2.0,False,93.219258,0.532566,0.171521,0.302661,0.369080,116.298580,118.802941,-2.504362
51331,42500311,2026-05-18 20:30:00,Thunder,1,False,1767.2500,2026-05-18,2026-05-11,7.0,False,...,3.0,False,96.995878,0.575412,0.138663,0.280188,0.304428,125.068780,108.302345,16.766435
51332,42500301,2026-05-19 20:00:00,Knicks,1,True,1741.2900,2026-05-19,2026-05-10,9.0,False,...,2.0,False,93.962230,0.539479,0.160889,0.304197,0.396093,118.509976,114.766564,3.743412
51333,42500312,2026-05-20 20:30:00,Thunder,1,True,1756.8900,2026-05-20,2026-05-18,2.0,False,...,2.0,False,99.391100,0.560754,0.148109,0.288533,0.304001,122.463248,107.590012,14.873236


In [113]:
difference_features = ['days_rest', 'possessions_rolling', 'eFG_rolling', 'TO%_rolling',
       'OREB%_rolling', 'FTR_rolling', 'off_rating_rolling', 'def_rating_rolling', 'net_rating_rolling', "pre_game_elo"]

for features in difference_features:
    game_df[f"{features}_diff"] = game_df[f"{features}_home"] - game_df[f"{features}_away"]

In [114]:
game_df = game_df.rename(columns={"gameDateTimeEst_home":"game_date"})
final_df = game_df[['game_date','pre_game_elo_home', 'is_B2B_home',
       'pre_game_elo_away', 'is_B2B_away', 'pre_game_elo_diff','days_rest_diff','possessions_rolling_diff', 'eFG_rolling_diff',
       'TO%_rolling_diff', 'OREB%_rolling_diff', 'FTR_rolling_diff',
       'off_rating_rolling_diff', 'def_rating_rolling_diff',
       'net_rating_rolling_diff', 'win_home']]

In [115]:
final_df.columns

Index(['game_date', 'pre_game_elo_home', 'is_B2B_home', 'pre_game_elo_away',
       'is_B2B_away', 'pre_game_elo_diff', 'days_rest_diff',
       'possessions_rolling_diff', 'eFG_rolling_diff', 'TO%_rolling_diff',
       'OREB%_rolling_diff', 'FTR_rolling_diff', 'off_rating_rolling_diff',
       'def_rating_rolling_diff', 'net_rating_rolling_diff', 'win_home'],
      dtype='str')

In [116]:
final_df = final_df.dropna()

In [117]:
final_df.shape

(51315, 16)

In [118]:
final_df

,game_date,pre_game_elo_home,is_B2B_home,pre_game_elo_away,is_B2B_away,pre_game_elo_diff,days_rest_diff,possessions_rolling_diff,eFG_rolling_diff,TO%_rolling_diff,OREB%_rolling_diff,FTR_rolling_diff,off_rating_rolling_diff,def_rating_rolling_diff,net_rating_rolling_diff,win_home
9,1986-11-01 20:00:00,1421.87,True,1448.97,True,-27.10,0.0,-10.982400,-0.099275,-0.086457,-0.091463,-0.002415,-8.464992,-3.910563,-4.554429,True
10,1986-11-01 20:00:00,1431.30,True,1518.94,True,-87.64,0.0,-8.064000,0.094880,-0.027357,-0.040309,-0.089677,8.659146,-2.711930,11.371076,True
12,1986-11-01 20:00:00,1471.80,True,1463.65,True,8.15,0.0,0.998400,0.161866,0.007656,-0.051724,-0.095930,18.519900,2.888894,15.631006,True
14,1986-11-01 20:00:00,1443.05,True,1524.34,True,-81.29,0.0,4.992000,-0.044118,0.040420,-0.180180,0.058229,-16.170137,16.614965,-32.785102,True
16,1986-11-01 20:00:00,1496.85,True,1461.72,True,35.13,0.0,4.723200,-0.195382,0.000538,-0.018490,-0.147186,-36.429595,-28.609532,-7.820063,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51330,2026-05-17 20:00:00,1691.98,False,1611.95,False,80.03,0.0,-1.397525,0.013263,-0.015842,0.029910,-0.108548,3.400311,-5.512977,8.913288,False
51331,2026-05-18 20:30:00,1767.25,False,1780.68,False,-13.43,4.0,-3.631092,0.022335,-0.018692,-0.014943,-0.069117,4.429258,6.834623,-2.405365,False
51332,2026-05-19 20:00:00,1741.29,False,1647.27,False,94.02,7.0,-1.847118,0.069457,-0.030585,0.001699,-0.078663,13.819604,-3.936978,17.756583,True
51333,2026-05-20 20:30:00,1756.89,False,1791.04,False,-34.15,0.0,-3.040712,0.017421,-0.026765,-0.041763,-0.077271,2.535078,6.817632,-4.282554,True
